In [ ]:
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langchain_huggingface import HuggingFaceEmbeddings

d:\ProdRAG\prodRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [4]:
load_dotenv()

True

In [ ]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.7)

In [ ]:
import os

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [ ]:
index_name = "practice-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [14]:
file_path = "D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()
print(type(loader))

<class 'langchain_community.document_loaders.pdf.PyPDFLoader'>


In [15]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [18]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
query = "What is the module4?"
results = retriever.invoke(query)

In [19]:
print(results)

[Document(id='69344ea5-8a65-4b39-b99b-f69757a7fda9', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30', 'creator': 'Microsoft® Word LTSC', 'moddate': '2025-10-10T09:37:45+05:30', 'page': 0.0, 'page_label': '1', 'producer': 'Microsoft® Word LTSC', 'source': 'D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf', 'total_pages': 4.0}, page_content='blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.'), Document(id='4a015367-b7d4-4779-8154-99c4f1c2c39c', metadata={'author': 'python-docx', 'creationdate': '2025-10-10T09:37:45+05:30',

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [ ]:
rag_chain = RunnableParallel(
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_docs),
    }
)

In [23]:
rag_chain.invoke("What is 4th module?")

{'question': 'What is 4th module?',
 'context': 'blockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \nand training data provenance.\n\nblockchain technology. \nModule 2: Cryptography Fundamentals — Basics of hashing, public and private key \nencryption, and digital signatures. \nModule 3: Consensus Algorithms — Understanding Proof of Work, Proof of Stake, and \nByzantine Fault Tolerance. \nModule 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using \nSolidity and Remix IDE. \nModule 5: Blockchain and AI Integration — Using blockchain to secure AI model sharing \n

In [ ]:
parser = StrOutputParser()

In [ ]:
output = rag_chain | rag_prompt | llm | parser

In [ ]:
output.invoke("What is 4th module?", config={"callbacks": [langfuse_handler]})

'Module 4: Smart Contracts and Ethereum — Writing and deploying smart contracts using Solidity and Remix IDE.'

Failed to export span batch code: None, reason: HTTPSConnectionPool(host='us.cloud.langfuse.com', port=443): Read timed out. (read timeout=5.0)
